# Generate Realistic Synthetic Dataset

Generates synthetic classification datasets with **non-linear label dependence**,
**latent factor structure**, and **realistic feature distributions** for XGBoost
scaling experiments.

Unlike `generate_imbalanced_data.ipynb` (which produces trivially separable data
with AUC=1.0), this notebook targets AUC-ROC ~0.80–0.85 with standard XGBoost.

**Tables produced:** `realistic_10k`, `realistic_1m`, `realistic_10m`, etc.

**Parameters** (via widgets or job params):
- `data_size`: Preset name (tiny/small/medium/medium_large/large/xlarge)
- `noise_scale`: Controls classification difficulty (higher = harder, default 2.0)
- `n_latent_factors`: Number of latent concepts driving feature clusters (default 6)
- `catalog`, `schema`: Unity Catalog location

## Setup Widgets

In [ ]:
dbutils.widgets.dropdown("data_size", "tiny", ["tiny", "small", "medium", "medium_large", "large", "xlarge"], "Data Size")
dbutils.widgets.text("noise_scale", "2.0", "Noise Scale (difficulty)")
dbutils.widgets.text("n_latent_factors", "6", "Latent Factors")
dbutils.widgets.text("seed", "42", "Random Seed")
dbutils.widgets.text("catalog", "brian_gen_ai", "Catalog")
dbutils.widgets.text("schema", "xgb_scaling", "Schema")
dbutils.widgets.dropdown("run_mode", "full", ["full", "smoke"], "Run Mode")

## Configuration

In [ ]:
import sys, os, time, json

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
repo_root = "/".join(notebook_path.split("/")[:-2])
sys.path.insert(0, f"/Workspace{repo_root}")

from src.config import REALISTIC_PRESETS

data_size = dbutils.widgets.get("data_size")
noise_scale = float(dbutils.widgets.get("noise_scale"))
n_latent_factors = int(dbutils.widgets.get("n_latent_factors"))
seed = int(dbutils.widgets.get("seed"))
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
run_mode = dbutils.widgets.get("run_mode")

preset = REALISTIC_PRESETS[data_size]
output_table = f"{catalog}.{schema}.{preset.table_name}"

# Smoke mode: override to tiny for quick validation
if run_mode == "smoke":
    total_rows = 1000
    n_numerical = 15
    n_categorical = 5
else:
    total_rows = preset.rows
    n_numerical = preset.num_features
    n_categorical = preset.cat_features

minority_ratio = preset.imbalance_ratio
n_features = n_numerical + n_categorical

print(f"Preset: {data_size} | Table: {output_table}")
print(f"Rows: {total_rows:,} | Numerical: {n_numerical} | Categorical: {n_categorical}")
print(f"Noise scale: {noise_scale} | Latent factors: {n_latent_factors} | Seed: {seed}")
print(f"Minority ratio: {minority_ratio:.1%}")

## Generate Dataset

In [ ]:
from pyspark.sql.functions import (
    rand, randn, when, col, lit, floor, abs as spark_abs,
    concat_ws, exp as spark_exp, log as spark_log, greatest,
    sin as spark_sin, cos as spark_cos,
    array, struct, udf
)
from pyspark.sql.types import FloatType, IntegerType, StringType
import pyspark.sql.functions as F
import math

# Categorical configs — same cardinality mix as original generator
CATEGORICAL_CONFIGS = [
    {"name": "binary",      "cardinality": 2,   "prefix": "cat_bin"},
    {"name": "low_card",    "cardinality": 5,   "prefix": "cat_low"},
    {"name": "medium_card", "cardinality": 20,  "prefix": "cat_med"},
    {"name": "high_card",   "cardinality": 100, "prefix": "cat_hi"},
    {"name": "very_high",   "cardinality": 500, "prefix": "cat_vhi"},
]


def _calibrate_bias(n_latent, noise_scale, minority_ratio, seed=42, n_sim=200_000):
    """
    Numerically calibrate the logit bias to achieve the target minority_ratio.
    
    The non-linear signal has a positive mean (~0.74) that analytical formulas
    can't capture accurately. Instead, simulate the logit distribution and
    binary-search for the bias that gives the correct minority fraction.
    """
    import numpy as np
    rng = np.random.default_rng(seed)
    z = [rng.standard_normal(n_sim) for _ in range(max(n_latent, 6))]
    
    signal = (
        0.8 * z[0] +
        0.5 * z[0]**2 +
        0.7 * z[min(1, n_latent-1)] * z[min(2, n_latent-1)] +
        0.6 * np.maximum(z[min(3, n_latent-1)], 0) +
        0.4 * np.sin(z[min(4, n_latent-1)] * 2.0)
    )
    if n_latent >= 6:
        signal = signal + 0.3 * z[5]**3
    
    noise = rng.standard_normal(n_sim) * noise_scale
    logit = signal + noise
    
    # Binary search for bias
    lo, hi = -15.0, 5.0
    for _ in range(60):
        mid = (lo + hi) / 2
        prob = 1.0 / (1.0 + np.exp(-(logit + mid)))
        ratio = prob.mean()
        if ratio > minority_ratio:
            hi = mid
        else:
            lo = mid
    
    bias = (lo + hi) / 2
    final_prob = 1.0 / (1.0 + np.exp(-(logit + bias)))
    actual_ratio = final_prob.mean()
    print(f"  Bias calibration: bias={bias:.3f} -> minority_ratio={actual_ratio:.4f} (target={minority_ratio:.4f})")
    return bias


def generate_realistic_dataset(
    spark,
    total_rows: int,
    n_numerical: int,
    n_categorical: int,
    n_latent: int,
    minority_ratio: float,
    noise_scale: float,
    seed: int,
):
    """
    Generate a realistic synthetic classification dataset.

    Key differences from generate_imbalanced_data:
    1. Latent factor structure — correlated feature clusters
    2. Non-linear label generation — interactions, quadratics, thresholds
    3. Realistic distributions — log-normal, zero-inflated, bimodal
    4. ~40% noise features uncorrelated with label

    Calibrated to produce AUC-ROC ~0.74–0.85 depending on feature count:
    - 15 features (tiny): ~0.74
    - 80 features (small): ~0.79
    - 200 features (medium): ~0.81
    - 400 features (large): ~0.81
    """
    n_partitions = max(8, total_rows // 2_000_000)

    print(f"Generating realistic dataset: {total_rows:,} rows")
    print(f"  Numerical: {n_numerical} | Categorical: {n_categorical}")
    print(f"  Latent factors: {n_latent} | Noise scale: {noise_scale}")
    print(f"  Partitions: {n_partitions}")

    # ── Step 1: Base dataframe with id ──
    df = spark.range(0, total_rows, numPartitions=n_partitions)

    # ── Step 2: Generate latent factors z0..z{n_latent-1} ──
    latent_cols = []
    for k in range(n_latent):
        col_name = f"_z{k}"
        latent_cols.append(randn(seed + k).cast(FloatType()).alias(col_name))
    df = df.select("*", *latent_cols)
    print(f"  Generated {n_latent} latent factors")

    # ── Step 3: Non-linear label logit from latent factors ──
    z = [col(f"_z{k}") for k in range(n_latent)]

    logit_terms = [
        lit(0.8) * z[0],                                    # linear
        lit(0.5) * z[0] * z[0],                             # quadratic
        lit(0.7) * z[min(1, n_latent-1)] * z[min(2, n_latent-1)],  # interaction
        lit(0.6) * greatest(z[min(3, n_latent-1)], lit(0.0)),       # ReLU
        lit(0.4) * spark_sin(z[min(4, n_latent-1)] * lit(2.0)),     # periodic
    ]
    if n_latent >= 6:
        logit_terms.append(lit(0.3) * z[5] * z[5] * z[5])  # cubic

    logit_expr = logit_terms[0]
    for term in logit_terms[1:]:
        logit_expr = logit_expr + term

    # Add Gaussian noise to logit (controls difficulty)
    logit_expr = logit_expr + randn(seed + 999).cast(FloatType()) * lit(noise_scale)

    # Numerically calibrate bias for target minority_ratio.
    # The signal has a positive mean (~0.74) from quadratic and ReLU terms,
    # so analytical log-odds formulas are inaccurate. Binary search is robust.
    bias = _calibrate_bias(n_latent, noise_scale, minority_ratio, seed)
    logit_expr = logit_expr + lit(bias)

    # sigmoid
    prob_expr = lit(1.0) / (lit(1.0) + spark_exp(-logit_expr))

    # Bernoulli label
    df = df.withColumn("_logit", logit_expr.cast(FloatType()))
    df = df.withColumn("_prob", prob_expr.cast(FloatType()))
    df = df.withColumn(
        "label",
        when(rand(seed + 888) < col("_prob"), lit(1)).otherwise(lit(0)).cast(IntegerType())
    )
    print("  Generated non-linear label")

    # ── Step 4: Generate numerical features in batches ──
    BATCH_SIZE = 50
    n_informative = max(1, int(n_numerical * 0.6))  # ~60% informative

    DIST_TYPES = ["gaussian", "lognormal", "zero_inflated", "bimodal", "heavy_tail"]

    print(f"  Generating {n_numerical} numerical features ({n_informative} informative)...")

    for batch_start in range(0, n_numerical, BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE, n_numerical)
        new_cols = []

        for i in range(batch_start, batch_end):
            feature_seed = seed + i + 100
            raw_noise = randn(feature_seed).cast(FloatType())

            if i < n_informative:
                latent_idx = i % n_latent
                loading = 0.7 + (i % 5) * 0.05  # 0.70–0.90 (was 0.3–0.7)
                loading = min(loading, 1.0)
                base = (
                    lit(loading) * col(f"_z{latent_idx}") +
                    lit(1.0 - loading) * raw_noise
                )
            else:
                base = raw_noise

            dist_type = DIST_TYPES[i % len(DIST_TYPES)]

            if dist_type == "gaussian":
                transformed = base
            elif dist_type == "lognormal":
                # Gentler: exp(x*0.25) instead of exp(x*0.5) to preserve more signal
                transformed = spark_exp(base * lit(0.25))
            elif dist_type == "zero_inflated":
                # Gentler: 15% zeros instead of 30%
                transformed = when(
                    rand(feature_seed + 5000) < lit(0.15), lit(0.0)
                ).otherwise(spark_abs(base))
            elif dist_type == "bimodal":
                # Gentler: shift ±1.0 instead of ±2.0
                transformed = base + when(
                    rand(feature_seed + 6000) < lit(0.5), lit(1.0)
                ).otherwise(lit(-1.0))
            elif dist_type == "heavy_tail":
                # Gentler: scale 0.15 instead of 0.5
                transformed = base * spark_abs(base) * lit(0.15)

            new_cols.append(transformed.cast(FloatType()).alias(f"f{i}"))

        df = df.select("*", *new_cols)

        if (batch_end) % 200 == 0 and batch_end < n_numerical:
            print(f"    Checkpointing at feature {batch_end}...")
            df = df.localCheckpoint(eager=True)

    print(f"  Numerical features done")

    # ── Step 5: Generate categorical features ──
    if n_categorical > 0:
        n_informative_cat = max(1, n_categorical // 5)
        print(f"  Generating {n_categorical} categorical features ({n_informative_cat} informative)...")

        for batch_start in range(0, n_categorical, BATCH_SIZE):
            batch_end = min(batch_start + BATCH_SIZE, n_categorical)
            new_cols = []

            for cat_idx in range(batch_start, batch_end):
                cfg = CATEGORICAL_CONFIGS[cat_idx % len(CATEGORICAL_CONFIGS)]
                cardinality = cfg["cardinality"]
                prefix = cfg["prefix"]
                feature_seed = seed + n_numerical + cat_idx + 2000
                col_name = f"{prefix}_{cat_idx}"

                if cat_idx < n_informative_cat:
                    latent_idx = cat_idx % n_latent
                    cat_int = (
                        floor(
                            spark_abs(
                                randn(feature_seed) + col(f"_z{latent_idx}") * lit(0.5)
                            ) * lit(cardinality / 3.0)
                        ) % lit(cardinality)
                    ).cast(IntegerType())
                else:
                    cat_int = floor(rand(feature_seed) * lit(cardinality)).cast(IntegerType())

                new_cols.append(
                    concat_ws("_", lit(prefix), cat_int.cast(StringType())).alias(col_name)
                )

            df = df.select("*", *new_cols)

        print(f"  Categorical features done")

    # ── Step 6: Final column selection (drop latent factors and intermediates) ──
    numerical_cols = [f"f{i}" for i in range(n_numerical)]
    categorical_cols = []
    for cat_idx in range(n_categorical):
        cfg = CATEGORICAL_CONFIGS[cat_idx % len(CATEGORICAL_CONFIGS)]
        categorical_cols.append(f"{cfg['prefix']}_{cat_idx}")

    all_cols = numerical_cols + categorical_cols + ["label"]
    df = df.select(all_cols)

    # Repartition for Delta write
    target_partitions = max(n_partitions, 200)
    if total_rows >= 50_000_000:
        target_partitions = max(n_partitions, 400)
    print(f"  Repartitioning to {target_partitions} for Delta write...")
    df = df.repartition(target_partitions)

    return df


start_time = time.time()

df = generate_realistic_dataset(
    spark=spark,
    total_rows=total_rows,
    n_numerical=n_numerical,
    n_categorical=n_categorical,
    n_latent=n_latent_factors,
    minority_ratio=minority_ratio,
    noise_scale=noise_scale,
    seed=seed,
)

gen_time = time.time() - start_time
print(f"\nDataFrame created in {gen_time:.1f}s (lazy)")

## Write to Delta Table

In [ ]:
print(f"Writing to: {output_table}")

write_start = time.time()

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(output_table)

write_time = time.time() - write_start
total_time = time.time() - start_time

print(f"Write completed in {write_time:.1f}s")
print(f"Total time: {total_time:.1f}s ({total_time / 60:.1f} minutes)")

## Validate Results

In [ ]:
df_check = spark.table(output_table)

row_count = df_check.count()
print(f"Rows written: {row_count:,}")
print(f"Expected:     {total_rows:,}")
print(f"Match: {row_count == total_rows}")

# Class distribution
label_counts = df_check.groupBy("label").count().orderBy("label").collect()
print("\nClass distribution:")
class_distribution = {}
for row in label_counts:
    label_val = row["label"]
    count = row["count"]
    pct = count / row_count * 100
    class_name = "Minority" if label_val == 1 else "Majority"
    print(f"  Label {label_val} ({class_name}): {count:,} ({pct:.2f}%)")
    class_distribution[label_val] = count

if len(label_counts) == 2:
    print(f"\nImbalance ratio: {label_counts[0]['count'] / label_counts[1]['count']:.1f}:1")

# Quick sample
sample_cols = [f"f{i}" for i in range(min(5, n_numerical))] + ["label"]
print(f"\nSample data:")
df_check.select(sample_cols).show(5, truncate=False)

## Quick XGBoost Sanity Check (tiny only)

For the `tiny` preset, train a quick XGBoost model to verify the data
produces realistic AUC (not 1.0). Skipped for larger presets.

In [ ]:
if total_rows <= 100_000:
    # Install XGBoost if not available (non-ML runtimes don't have it)
    try:
        import xgboost as xgb
    except ImportError:
        import subprocess
        subprocess.check_call(["pip", "install", "-q", "xgboost", "scikit-learn"])
        import xgboost as xgb

    import numpy as np
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score, average_precision_score

    pdf = df_check.toPandas()
    feature_cols = [c for c in pdf.columns if c != "label" and not c.startswith("cat_")]
    X = pdf[sorted(feature_cols)].values.astype(np.float32)
    y = pdf["label"].values

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    neg, pos = np.sum(y_train == 0), np.sum(y_train == 1)
    spw = neg / max(pos, 1)

    model = xgb.XGBClassifier(
        tree_method="hist", n_estimators=200, max_depth=8,
        learning_rate=0.1, scale_pos_weight=spw, random_state=42, verbosity=0
    )
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]

    auc_roc = roc_auc_score(y_test, y_proba)
    auc_pr = average_precision_score(y_test, y_proba)

    print(f"\n--- XGBoost sanity check (numeric features only) ---")
    print(f"AUC-ROC: {auc_roc:.4f}")
    print(f"AUC-PR:  {auc_pr:.4f}")
    if auc_roc > 0.95:
        print(f"WARNING: AUC-ROC {auc_roc:.3f} > 0.95 — data may be too easy. Increase noise_scale.")
    elif auc_roc < 0.60:
        print(f"WARNING: AUC-ROC {auc_roc:.3f} < 0.60 — data may be too hard. Decrease noise_scale.")
    else:
        print(f"AUC in target range (0.60–0.95). Looks realistic.")
else:
    print(f"Skipping XGBoost sanity check (dataset too large: {total_rows:,} rows)")
    auc_roc = None
    auc_pr = None

## Exit with Result

In [ ]:
result = {
    "status": "ok",
    "table": output_table,
    "data_size": data_size,
    "total_rows": total_rows,
    "n_numerical": n_numerical,
    "n_categorical": n_categorical,
    "n_latent_factors": n_latent_factors,
    "noise_scale": noise_scale,
    "seed": seed,
    "minority_ratio": minority_ratio,
    "actual_rows": row_count,
    "class_distribution": {str(k): v for k, v in class_distribution.items()},
    "total_time_sec": round(total_time, 1),
}
if auc_roc is not None:
    result["sanity_auc_roc"] = round(auc_roc, 4)
    result["sanity_auc_pr"] = round(auc_pr, 4)

result_json = json.dumps(result)
print(f"\nResult: {result_json}")
dbutils.notebook.exit(result_json)